# Chapter 5. EDA (Exploratory Data Analysis)

데이터를 분석하는 과정으로 모델링을 하기 전에 데이터와 친해지고 어떤 정보를 가지고 있고, 분포는 어떻게 생겼는지 알아가는 과정이다. 데이터를 분석할 때는 아래의 5가지로 나눌 수 있는데, 

1. 문제 정의 
2. 데이터 수집 
3. 데이터 pre-processing 
4. EDA 
5. modeling 

이번 수업에서 우리는 EDA 파트에 대해서 배울 예정이다.

## 한글 폰트 설치 및 설정 

그래프에 그래프제목, X축이름, Y축이름 등 한글이 들어가는 경우가 많으므로 한글이 깨지지않도록 폰트 설치 및 설정을 먼저 해주자.

````admonition {한글 폰트 설치 및 설정}
:class: dropdown 

* Step1: 아래 셀을 실행시켜 폰트를 설치해준다. 
```bash
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf
```
* Step2: Colab 메뉴에서 런타임 -> 세션 다시 시작을 해준다. 
* Step3: 폰트를 설정한다. 

```python
import matplotlib.pyplot as plt
import numpy as np

plt.rc('font', family='NanumBarunGothic')
```
* Step4. 폰트 설정이 잘 되었는지 확인해본다. '한글'이라는 글자가 출력이 되었다면 잘된것이다. 
```python
plt.text(0.3, 0.3, '한글', size=100)
plt.show()
```
````

## EDA 

우리가 가진 데이터를 본격적으로 분석하기 전에 **그 데이터를 탐색하고 이해하는 과정**을 말합니다. 
EDA는 데이터의 평균, outlier 등에 대해서 데이터 자체의 기본적인 정보를 파악하고 다음 파트 modeling을 위해 데이터를 알아가는 과정이다. 

EDA에서는 다음과 같은 작업을 한다. 

| # | 분석 항목 | 설명 | 주요 함수 |
|---|-----------|------|-----------|
| 1 | 기본 정보 확인 | 컬럼 데이터 타입, column/index 정보 파악 | `df.info()`, `df.dtypes`, `df.shape`, `df.columns` |
| 2 | 대표값 확인 | 데이터의 '평균적인 얼굴': 평균·중앙값·최빈값 | `df.mean()`, `df.median()`, `df.mode()` |
| 3 | 데이터 건강도 확인 | 결측치(NaN)와 이상치(Outlier) 탐지 | `df.isnull().sum()`, `df.describe()` |
| 4 | 산포도 | 데이터가 평균 주위에 얼마나 퍼져 있는지 파악 | `df.std()`, `df.var()` |
| 5 | 분포 확인 | 사분위수로 분포 형태와 치우침 파악 | `df.describe()`, `df.quantile([.25, .5, .75])` |
| 6 | 데이터 요약·변형 | Pivot/Unpivot으로 데이터 재구조화 | `df.pivot_table()`, `pd.melt()` |

이번 실습에서는 공개 교통(자동차) 데이터를 활용하여 위의 EDA 과정을 단계적으로 직접 실습합니다.

사용 데이터: **Auto MPG 데이터셋** — 미국·유럽·일본산 자동차(1970~1982) 398대의 연비, 배기량, 출력 등 성능 데이터  
- 출처: UCI Machine Learning Repository (Seaborn 내장 제공)  
- 특징: 수치형·범주형 데이터가 혼합되어 있고, `horsepower` 컬럼에 실제 결측치가 존재하여 결측치 실습에 적합합니다.

| 컬럼 | 설명 | 타입 |
|------|------|------|
| mpg | 연비 (miles per gallon) | 수치형(연속) |
| cylinders | 실린더 수 | 수치형(이산) |
| displacement | 배기량 | 수치형(연속) |
| horsepower | 출력 (마력) | 수치형(연속), **결측치 존재** |
| weight | 차량 무게 | 수치형(연속) |
| acceleration | 가속도 | 수치형(연속) |
| model_year | 모델 연도 | 수치형(이산) |
| origin | 생산 국가 (usa/japan/europe) | 범주형(명목) |
| name | 차량 이름 | 범주형(명목) |

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 한글 폰트 설정 (Colab에서 폰트 설치 후 주석 해제)
# plt.rc('font', family='NanumBarunGothic')

# Seaborn 내장 자동차(교통) 데이터셋 불러오기
df = sns.load_dataset('mpg')
df.head()

### EDA: 기본 정보 확인 

#### 데이터 크기 확인
* 몇 개의 행(Row) 으로 이루어져 있나요?
  * 행은 보통 하나의 관측치 (Observation) 또는 개별 기록을 나타낸다. 행의 수가 많으면 데이터가 방대하다는 뜻이다. 
* 몇 개의 열(Column)으로 이루어져 있나요? 
  * 열은 데이터의 변수(Variable) 또는 특성(Feature)을 나타낸다. 열의 수가 많으면 파악해야할 정보가 많다는 뜻이다. 

In [ ]:
# 데이터 크기 확인
print("데이터 크기 (행 × 열):", df.shape)
print(f"  → 관측치(행): {df.shape[0]}개  |  변수(열): {df.shape[1]}개")

#### 데이터 내용 (샘플) 확인

데이터의 실제 모습을 직접 보면서 직관적으로 이해하고, 혹시 이상한 부분이 없는지 체크한다. 

* 앞부분 확인: 데이터가 어떤 식으로 구성되어 있는지, 각 열에 어떤 종류의 값이 들어있는지 확인 
* 뒷부분 확인: 때로는 데이터의 마지막 부분에 요약 정보나 이상한 값들이 포함되어 있을 수 있어, 이를 미리 발견 가능 
  

In [ ]:
# 앞부분 5행 확인
print("=== 앞부분 (head) ===")
display(df.head())

# 뒷부분 5행 확인
print("\n=== 뒷부분 (tail) ===")
display(df.tail())

#### 고유값 (Unique Value) 확인 

고유값은 특정 컬럼 내에서 중복되지 않는 값들을 의미한다. 즉, 여러 번 나타나더라도 해당 컬럼에 어떤 종류의 값들이 존재하는지 파악할 때 사용된다. 

고유값이 중요한 이유:
* 데이터 탐색 및 이해: 특정 컬럼에 어떤 종류의 범주가 있는지 빠르게 파악할 수 있습니다. 예를 들어, '지역' 컬럼의 고유값을 확인하면 데이터에 포함된 모든 지역 목록을 알 수 있습니다.
* 데이터 품질 확인: 오타나 일관성 없는 데이터 입력(예: '서울', 'seoul', '서울시')을 고유값을 통해 쉽게 발견할 수 있습니다.
* 범주형 변수 처리: 머신러닝 모델 학습 등에서 범주형 변수를 원-핫 인코딩(One-Hot Encoding) 등으로 변환할 때 고유값의 개수를 알아야 합니다.

In [ ]:
# origin 컬럼의 고유값 — 어느 나라 차량이 포함되어 있나?
print("origin 고유값:", df['origin'].unique())
print("origin 고유값 개수:", df['origin'].nunique())

# cylinders 고유값 — 어떤 실린더 수가 존재하나?
print("\ncylinders 고유값:", sorted(df['cylinders'].unique()))

#### 데이터 타입 확인 

각 컬럼의 데이터 타입이 올바르게 설정되어 있는지 확인하는 것이 중요합니다. 예를 들어, '구매금액'이 숫자가 아니라 글자로 인식되어 있다면, 나중에 합계나 평균을 계산할 수 없게 되죠. 데이터 타입이 잘못되어 있으면 데이터 분석 전에 반드시 수정해야 합니다.

* 수치형 데이터: 숫자로 표현되는 정보, 이 숫자들은 크고 작음을 비교할 수 있고, 더하거나 빼는 등의 수학적인 계산이 가능합니다. 키, 몸무게, 매출액, 온도, 판매량 등이 여기에 해당해요.
  * 특징: 
    * 계산 가능: 평균, 합계, 최댓값, 최솟값 등을 구할 수 있습니다.
    * 크기 비교 가능: "이 값이 저 값보다 크다/작다"를 판단할 수 있습니다.
  * 연속형 데이터 (Continuous): 소수점을 포함하는 무한히 세밀한 값. 예) 키(170.5cm), 온도(36.7°C), 연비(25.4 mpg)
  * 이산형 데이터 (Discrete): 정수만 가능한 값. 예) 실린더 수(4, 6, 8), 판매 개수
* 범주형 데이터: 분류를 위한 정보, 범주형 데이터는 숫자보다는 어떤 그룹이나 카테고리를 나타내는 데이터를 말합니다. 이 데이터들은 서로 다른 그룹으로 분류되지만, 숫자의 크기처럼 더하거나 빼는 계산은 의미가 없습니다. 성별, 혈액형, 지역, 만족도 등이 대표적이에요.
  * 특징: 
    * 분류 가능: 데이터를 몇 개의 그룹으로 나눌 수 있습니다.
    * 계산 불가능: 사칙연산은 의미가 없습니다. (성별 '남' + '여'가 무슨 의미일까요?)
  * 명목형 데이터 (Nominal): 순서가 의미 없음. 예) 원산지('usa', 'japan', 'europe'), 혈액형
  * 순서형 데이터 (Ordinal): 순서에 의미가 있음. 예) 만족도(1~5점), 학년
* 날짜/시간 데이터: 시간의 흐름을 나타내는 정보 
  * 시간의 흐름: 순서대로 정렬할 수 있고, 기간을 계산할 수 있습니다.
  * 다양한 형식: 연/월/일, 시/분/초 등 다양한 형태로 표현될 수 있습니다.

#### 데이터 타입별 활용 방법

데이터 타입에 따라 적합한 분석 방법과 시각화가 달라집니다.

| 데이터 타입 | 대표 분석 | 대표 시각화 | 주요 함수 예시 |
|------------|-----------|------------|--------------|
| 수치형(연속) | 평균, 표준편차, 사분위수 | 히스토그램, 박스플롯, 산점도 | `df.describe()`, `df.corr()` |
| 수치형(이산) | 빈도수, 최빈값 | 막대그래프, 히스토그램 | `df.value_counts()` |
| 범주형(명목) | 빈도수, 비율 | 막대그래프, 파이차트 | `df.value_counts()`, `df.groupby()` |
| 범주형(순서) | 빈도수, 누적 비율 | 순서 막대그래프 | `df.value_counts()` |
| 날짜/시간형 | 추세 분석, 기간 계산 | 라인차트(시계열) | `pd.to_datetime()`, `df.resample()` |

In [ ]:
# 각 컬럼의 데이터 타입 확인
print("=== df.dtypes (컬럼별 타입) ===")
print(df.dtypes)

print("\n=== df.info() (전체 요약) ===")
df.info()

### EDA: 대표값 확인

데이터 전체의 특징을 하나의 숫자로 요약하는 값을 **대푯값(Representative Value)**이라 합니다.  
같은 데이터라도 어떤 대푯값을 쓰느냐에 따라 해석이 달라질 수 있으므로, 세 가지를 함께 확인하는 것이 중요합니다.

| 대푯값 | 설명 | 특징 | 함수 |
|--------|------|------|------|
| **평균 (Mean)** | 모든 값의 합 ÷ 데이터 개수 | 이상치에 민감, 전체 합계를 반영 | `df.mean()` |
| **중앙값 (Median)** | 오름차순 정렬 후 가운데 위치의 값 | 이상치 영향을 받지 않음 | `df.median()` |
| **최빈값 (Mode)** | 가장 자주 등장하는 값 | 범주형 데이터에 유용, 여러 개 존재 가능 | `df.mode()` |

**어떤 대푯값을 써야 할까?**  
평균과 중앙값의 차이가 크다면 이상치가 있다는 신호입니다.  
예를 들어 소수의 고소득자가 있는 연봉 데이터는 평균이 중앙값보다 훨씬 높게 나타납니다. 이럴 때는 중앙값이 더 대표성을 갖습니다.

In [ ]:
# 수치형 컬럼의 대표값 확인
print("=== 평균 (mean) ===")
print(df.select_dtypes(include='number').mean().round(2))

print("\n=== 중앙값 (median) ===")
print(df.select_dtypes(include='number').median())

print("\n=== 최빈값 (mode) — origin(범주형) 컬럼 예시 ===")
print(df['origin'].mode())

### EDA: 데이터 건강도 확인 (결측치와 이상치)

종종 특정 값이 비어있는 칸이 있는데, 이를 결측치(Missing Values)라고 부릅니다. '정보 없음', '응답 없음', '측정 불가' 등 다양한 이유로 발생할 수 있습니다.

#### 결측치를 확인해야 하는 이유 

* **분석 오류 방지**: 결측치가 있는 상태로 분석하면 평균, 합계 등이 왜곡될 수 있습니다.
* **모델 학습 불가**: 대부분의 머신러닝 알고리즘은 결측치가 있으면 학습 자체가 되지 않습니다.
* **패턴 발견**: 결측치가 특정 조건에서만 발생한다면, 그것 자체가 중요한 정보일 수 있습니다.

#### 결측치 처리 방법 

* **방법 1) 데이터 제거**
  * 장점: 구현이 간단하고 빠릅니다.
  * 단점: 데이터 손실이 발생합니다. 결측치가 많은 경우 분석 가능한 샘플 수가 크게 줄어듭니다.
  * 행 제거: `df.dropna()` — 결측치가 하나라도 있는 행 삭제
  * 열 제거: `df.dropna(axis=1)` — 결측치가 있는 열 전체 삭제

* **방법 2) 결측치 채우기 (`df.fillna()`)**
  * **평균(mean)으로 채우기**: 수치형 데이터에 적합. 이상치가 없을 때 분포를 크게 왜곡하지 않습니다.
  * **중앙값(median)으로 채우기**: 이상치가 있는 수치형 데이터에 더 안정적입니다.
  * **최빈값으로 채우기**: 범주형 데이터(예: '성별', '혈액형')의 결측치에 적합합니다.
  * **이전/이후 값으로 채우기**: 시계열 데이터처럼 순서가 있는 경우 유용합니다. `df.fillna(method='ffill')`
  * **상수로 채우기**: `df.fillna(0)` 또는 `df.fillna('unknown')` 등 고정값으로 채웁니다.
  * **예측 모델로 채우기**: 다른 컬럼 정보를 활용해 머신러닝 모델로 결측치를 예측하여 채웁니다. 가장 정교하지만 구현이 복잡합니다.

In [ ]:
# 결측치 확인
print("=== 컬럼별 결측치 수 ===")
print(df.isnull().sum())

print("\n=== 결측치 비율 (%) ===")
print((df.isnull().sum() / len(df) * 100).round(2))

# horsepower에 결측치가 있음을 확인
print("\nhorsepower 결측치 행:")
display(df[df['horsepower'].isnull()])

# ────────────────────────────────────
# 방법 1: 결측치가 있는 행 제거
df_drop = df.dropna()
print(f"\n[방법1 행 제거] 제거 전: {len(df)}행 → 제거 후: {len(df_drop)}행")

# 방법 2-1: 평균값으로 채우기
df_mean = df.copy()
mean_hp = df_mean['horsepower'].mean()
df_mean['horsepower'] = df_mean['horsepower'].fillna(mean_hp)
print(f"[방법2-1 평균] horsepower 결측치를 평균 {mean_hp:.2f}로 대체")

# 방법 2-2: 중앙값으로 채우기
df_median = df.copy()
median_hp = df_median['horsepower'].median()
df_median['horsepower'] = df_median['horsepower'].fillna(median_hp)
print(f"[방법2-2 중앙값] horsepower 결측치를 중앙값 {median_hp:.1f}으로 대체")

#### 이상치 확인

데이터에서 다른 값들과 동떨어진 **이상치(Outlier)**는 분석 결과를 왜곡할 수 있습니다.  
이상치를 처리하기 전에 먼저 어떤 방법으로 탐지할지 결정해야 합니다.

##### 이상치 확인 방법

| 방법 | 설명 | 도구 |
|------|------|------|
| **시각적 확인** | 박스플롯, 산점도, 히스토그램으로 눈으로 확인 | `sns.boxplot()`, `plt.scatter()`, `plt.hist()` |
| **IQR 방법** | Q1 - 1.5×IQR 미만 또는 Q3 + 1.5×IQR 초과 | `df.quantile()` |
| **Z-score 방법** | 평균으로부터 ±3 표준편차 이상 벗어난 값 | `scipy.stats.zscore()` |

##### 이상치 처리 방법

| 방법 | 설명 | 적합한 상황 |
|------|------|-----------|
| **제거 (Removal)** | 이상치를 데이터셋에서 완전히 삭제 | 명백한 오류 데이터, 샘플 수가 충분할 때 |
| **변환 (Transformation)** | 로그·제곱근·Box-Cox 변환으로 분포 조정 | 분포를 유지하면서 이상치 영향 완화 |
| **대체 (Imputation)** | 중앙값, 경계값(Winsorizing)으로 대체 | 데이터 손실 없이 이상치 영향을 줄일 때 |
| **유지 (Keep as is)** | 이상치를 보존하되 별도 플래그 표시 | 도메인 지식상 의미 있는 극값인 경우 |

In [ ]:
# IQR 방법으로 이상치 탐지
Q1 = df['horsepower'].quantile(0.25)
Q3 = df['horsepower'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['horsepower'] < lower) | (df['horsepower'] > upper)]
print(f"IQR 기준 — 하한: {lower:.1f}  |  상한: {upper:.1f}")
print(f"이상치 수: {len(outliers)}개\n")
display(outliers[['name', 'horsepower', 'origin']].head(10))

### EDA: 산포도

같은 평균을 가진 두 데이터셋이라도 값들이 얼마나 퍼져 있는지는 완전히 다를 수 있습니다.  
이처럼 **데이터가 얼마나 퍼져 있는지**를 나타내는 지표를 **산포도(Measure of Dispersion)**라고 합니다.

| 지표 | 설명 | 장점 | 단점 | 함수 |
|------|------|------|------|------|
| **범위 (Range)** | 최댓값 − 최솟값 | 계산이 가장 간단 | 이상치에 극도로 민감 | `df.max() - df.min()` |
| **분산 (Variance)** | 각 값이 평균에서 벗어난 정도의 제곱 평균 | 편차를 수학적으로 정확히 표현 | 단위가 원본의 제곱이라 직관적 해석 어려움 | `df.var()` |
| **표준편차 (STD)** | 분산의 제곱근 | 원본과 단위가 같아 직관적 해석 가능 | 이상치의 영향을 받음 | `df.std()` |

**분산 vs 표준편차**  
키 데이터가 cm 단위라면, 분산의 단위는 cm²이 되어 "키의 분산이 116 cm²"처럼 해석하기 어렵습니다.  
표준편차는 분산에 √를 씌워 다시 cm 단위로 되돌리므로, "키의 표준편차가 10.77 cm"처럼 직관적으로 해석할 수 있습니다.

In [ ]:
# mpg(연비) 컬럼으로 산포도 지표 확인
mpg = df['mpg']

print(f"범위  (Range)    : {mpg.max() - mpg.min():.2f}")
print(f"분산  (Variance) : {mpg.var():.2f}")
print(f"표준편차 (STD)   : {mpg.std():.2f}")
print(f"최솟값: {mpg.min():.1f}  |  최댓값: {mpg.max():.1f}  |  평균: {mpg.mean():.2f}")

### EDA: 분포 (사분위수)

**사분위수(Quartile)**는 데이터를 오름차순으로 정렬한 후 4등분하는 값입니다.

| 사분위수 | 설명 | 백분위수 |
|---------|------|---------|
| Q1 (1사분위수) | 하위 25% 지점의 값 | 25th percentile |
| Q2 (2사분위수) | 중앙값 (Median) | 50th percentile |
| Q3 (3사분위수) | 상위 25% 지점의 값 | 75th percentile |
| **IQR** | Q3 − Q1 (사분위 범위) | 중간 50% 데이터의 폭 |

**IQR을 활용한 이상치 탐지**  
IQR을 사용하면 이상치의 경계를 수학적으로 정의할 수 있습니다.
* 하한 이상치: `Q1 - 1.5 × IQR` 보다 작은 값
* 상한 이상치: `Q3 + 1.5 × IQR` 보다 큰 값

`df.describe()`를 호출하면 min, Q1(25%), Q2(50%), Q3(75%), max를 한 번에 확인할 수 있습니다.

참고: https://wikidocs.net/285507

In [ ]:
# 전체 수치형 컬럼 요약 (min, 25%, 50%, 75%, max 포함)
print("=== df.describe() ===")
display(df.describe())

# mpg 컬럼의 사분위수 직접 계산
mpg = df['mpg']
Q1 = mpg.quantile(0.25)
Q2 = mpg.quantile(0.50)
Q3 = mpg.quantile(0.75)
IQR = Q3 - Q1

print(f"\nmpg 사분위수:")
print(f"  Q1 (25%): {Q1}  |  Q2 (50%): {Q2}  |  Q3 (75%): {Q3}  |  IQR: {IQR}")
print(f"이상치 기준: {Q1 - 1.5*IQR:.2f} 미만  또는  {Q3 + 1.5*IQR:.2f} 초과")

# 박스플롯으로 사분위수 시각화
df[['mpg', 'horsepower', 'weight']].plot(
    kind='box', subplots=True, layout=(1, 3), figsize=(12, 5),
    title=['mpg 연비', 'horsepower 출력', 'weight 무게']
)
plt.suptitle('수치형 컬럼 박스플롯 (사분위수 시각화)', y=1.02)
plt.tight_layout()
plt.show()

### EDA: 데이터 요약과 변형 (Pivot & Unpivot)

**Pivot(피벗)**은 행에 있는 데이터를 열로 바꾸어 요약 테이블을 만드는 것입니다.  
**Unpivot(언피벗)**은 그 반대로, 여러 열을 하나의 열로 녹여 긴 형태(Long Format)로 바꿉니다.

| 변환 | 방향 | 설명 | 함수 |
|------|------|------|------|
| **Pivot** | Wide 형태로 | 행 → 열, 집계 요약 테이블 생성 | `df.pivot_table(values, index, columns, aggfunc)` |
| **Unpivot** | Long 형태로 | 열 → 행, Wide → Long 변환 | `pd.melt(df, id_vars, value_vars)` |

**언제 사용하나요?**
- **Pivot**: "연도별·지역별 평균 매출"처럼 두 범주형 변수를 기준으로 집계할 때
- **Unpivot**: 시각화 라이브러리(seaborn 등)나 머신러닝 모델이 Long Format 데이터를 요구할 때

참고: https://wikidocs.net/216450

In [ ]:
# Pivot: cylinders × origin 별 평균 연비(mpg) 요약 테이블
pivot_table = df.pivot_table(
    values='mpg',
    index='cylinders',
    columns='origin',
    aggfunc='mean'
).round(2)

print("=== Pivot Table: 실린더 수 × 원산지 별 평균 연비(mpg) ===")
display(pivot_table)

# Unpivot (melt): 수치형 컬럼들을 Long Format으로 변환
df_wide = df[['mpg', 'horsepower', 'weight']].head(5)
df_long = pd.melt(df_wide, var_name='변수', value_name='값')

print("\n=== 원본 (Wide Format) ===")
display(df_wide)
print("\n=== melt 후 (Long Format) ===")
display(df_long)